# NB13 — Análisis de errores para el capítulo 4

Usa las predicciones guardadas por NB12 (contexto 7, cinco semillas). No entrena modelos. Ejecutar dentro del repositorio actualizado con los scripts del paquete. Las salidas se guardan en `reports/results/chapter4` y `reports/figures/chapter4`.

In [1]:
from pathlib import Path
import sys
import subprocess

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "scripts/error_analysis.py").exists():
    raise FileNotFoundError("Copiá primero el contenido del paquete en la raíz del repositorio.")
sys.path.insert(0, str(ROOT))
from scripts.error_analysis import run


## Matrices de confusión y errores

Se verifica que las cinco semillas compartan centros y etiquetas. La matriz temporal muestra la media de matrices normalizadas por clase real: no representa un ensamble. Se guardan precisión, recall, F1, errores en transiciones y ejemplos por par de clases. Los casos SN069 y SN081 se muestran para todas las semillas.

In [2]:
per_stage, transition_errors = run(ROOT)
display(per_stage.groupby("stage")[["precision", "recall", "f1"]].agg(["mean", "std"]))
display(transition_errors)

Resultados: C:\Users\tolot\Documents\UBA\2026\TP\sleep-staging-tp\reports\results\chapter4
Figuras: C:\Users\tolot\Documents\UBA\2026\TP\sleep-staging-tp\reports\figures\chapter4


precision              recall                  f1          
           mean       std      mean       std      mean       std
stage                                                            
N1     0.446432  0.006177  0.615344  0.014495  0.517318  0.002320
N2     0.871704  0.005156  0.745082  0.009692  0.803387  0.004543
N3     0.823481  0.008789  0.891141  0.007048  0.855923  0.002866
REM    0.848087  0.014926  0.894138  0.015465  0.870294  0.001661
W      0.860276  0.007236  0.797767  0.015644  0.827717  0.005409

,seed,is_transition,n,errors,error_rate
0,42,False,19400,2992,0.154227
1,42,True,7046,2683,0.380783
2,123,False,19400,2910,0.150000
3,123,True,7046,2693,0.382203
4,2026,False,19400,2803,0.144485
5,2026,True,7046,2697,0.382770
6,7,False,19400,2872,0.148041
7,7,True,7046,2690,0.381777
8,99,False,19400,2838,0.146289
9,99,True,7046,2709,0.384473


## Revisar las confusiones y ejemplos

Los ejemplos siguen un criterio fijo: primer error de cada par de clases por semilla, ordenado por sujeto e índice. Son descriptivos; no permiten atribuir causas fisiológicas sin inspeccionar las señales.

In [3]:
import pandas as pd
OUT = ROOT / "reports/results/chapter4"
display(pd.read_csv(OUT / "confusion_pairs.csv"))
display(pd.read_csv(OUT / "error_examples.csv").head(20))

,real,predicted,mean_count,mean_rate,std_rate
0,N2,N1,1252.8,0.123344,0.009382
1,N2,N3,933.4,0.091897,0.006324
2,W,N1,816.6,0.193967,0.014804
3,N3,N2,526.6,0.105068,0.006937
4,N1,N2,389.0,0.128638,0.013140
5,N1,W,385.0,0.127315,0.010769
6,N1,REM,371.0,0.122685,0.013649
7,N2,REM,253.2,0.024929,0.004510
8,REM,N1,224.8,0.055602,0.010553
9,REM,N2,191.0,0.047242,0.006248


,subject,source_epoch_index,y_true,y_pred,is_transition,seed
0,SN010,61,1,0,True,42
1,SN010,85,0,1,True,42
2,SN010,89,2,1,True,42
3,SN010,97,1,2,True,42
4,SN010,100,2,3,False,42
5,SN010,189,0,3,True,42
6,SN010,199,0,2,True,42
7,SN010,428,2,4,True,42
8,SN010,446,4,1,False,42
9,SN010,514,4,2,False,42


## Preparar archivos para continuar la memoria y verificar la API

Esta celda empaqueta pesos, normalización, predicciones y metadata. Incluye SN069 como ejemplo del test si está disponible. No necesita volver a entrenar. Subí el ZIP generado para completar la revisión con resultados reales.

In [4]:
sample = ROOT / "data/processed/SN069.npz"
command = [sys.executable, str(ROOT / "scripts/collect_chapter4_artifacts.py"), "--root", str(ROOT), "--out", str(ROOT / "chapter4_artifacts.zip")]
if sample.exists():
    command += ["--sample", str(sample)]
else:
    print("No se encontró SN069.npz; el ZIP no incluirá un registro para la prueba de la API.")
subprocess.run(command, check=True)

CompletedProcess(args=['c:\\Users\\tolot\\anaconda3\\envs\\sleep-staging\\python.exe', 'C:\\Users\\tolot\\Documents\\UBA\\2026\\TP\\sleep-staging-tp\\scripts\\collect_chapter4_artifacts.py', '--root', 'C:\\Users\\tolot\\Documents\\UBA\\2026\\TP\\sleep-staging-tp', '--out', 'C:\\Users\\tolot\\Documents\\UBA\\2026\\TP\\sleep-staging-tp\\chapter4_artifacts.zip', '--sample', 'C:\\Users\\tolot\\Documents\\UBA\\2026\\TP\\sleep-staging-tp\\data\\processed\\SN069.npz'], returncode=0)

## Ensayo de integración (después de iniciar la API)

Desde la raíz, iniciar `python -m uvicorn sleep_service.app:app --host 127.0.0.1 --port 8000`. Abrir http://127.0.0.1:8000. La prueba siguiente compara las predicciones de un NPZ de test con NB12 para la misma semilla y mide tres peticiones. Su alcance comienza en el NPZ de NB2, no en EDF crudo.

In [ ]:
RUN_API_TEST = True  # Cambiar a True cuando el servidor esté ejecutándose.
if RUN_API_TEST:
    subprocess.run([sys.executable, str(ROOT / "scripts/validate_api.py"), str(sample), "--root", str(ROOT)], check=True)
else:
    print("Ensayo de integración pendiente. No reportar tiempos ni concordancia hasta ejecutarlo.")

: 